In [1]:
import pandas as pd
import numpy as np

retail_df = pd.read_csv("../../02_Cleaned_Data/online_retail_cleaned.csv")

retail_df.shape

(779425, 8)

In [2]:
retail_df["Revenue"] = retail_df["Quantity"] * retail_df["Price"]

retail_df["InvoiceDate"] = pd.to_datetime(retail_df["InvoiceDate"])

retail_df[["Quantity","Price","Revenue"]].head()

,Quantity,Price,Revenue
0,12,6.95,83.4
1,12,6.75,81.0
2,12,6.75,81.0
3,48,2.10,100.8
4,24,1.25,30.0


In [3]:
snapshot_date = retail_df["InvoiceDate"].max() + pd.Timedelta(days=1)

customer_recency = retail_df.groupby("Customer ID")["InvoiceDate"] \
    .max() \
    .reset_index()

customer_recency["Recency"] = (
    snapshot_date - customer_recency["InvoiceDate"]
).dt.days

customer_recency.head()

,Customer ID,InvoiceDate,Recency
0,12346.0,2011-01-18 10:01:00,326
1,12347.0,2011-12-07 15:52:00,2
2,12348.0,2011-09-25 13:13:00,75
3,12349.0,2011-11-21 09:51:00,19
4,12350.0,2011-02-02 16:01:00,310


In [4]:
customer_frequency = retail_df.groupby("Customer ID")["Invoice"].nunique().reset_index()

customer_frequency.columns = [
    "Customer ID",
    "Purchase_Frequency"
]

customer_frequency.head()

,Customer ID,Purchase_Frequency
0,12346.0,12
1,12347.0,8
2,12348.0,5
3,12349.0,4
4,12350.0,1


In [5]:
customer_monetary = retail_df.groupby("Customer ID")["Revenue"] \
    .sum() \
    .reset_index()

customer_monetary.columns = [
    "Customer ID",
    "Monetary_Value"
]

customer_monetary.head()

,Customer ID,Monetary_Value
0,12346.0,77556.46
1,12347.0,4921.53
2,12348.0,2019.40
3,12349.0,4428.69
4,12350.0,334.40


In [6]:
rfm_df = customer_recency.merge(
    customer_frequency,
    on="Customer ID"
)

rfm_df = rfm_df.merge(
    customer_monetary,
    on="Customer ID"
)

rfm_df.head()

,Customer ID,InvoiceDate,Recency,Purchase_Frequency,Monetary_Value
0,12346.0,2011-01-18 10:01:00,326,12,77556.46
1,12347.0,2011-12-07 15:52:00,2,8,4921.53
2,12348.0,2011-09-25 13:13:00,75,5,2019.40
3,12349.0,2011-11-21 09:51:00,19,4,4428.69
4,12350.0,2011-02-02 16:01:00,310,1,334.40


In [7]:
rfm_df.shape

(5878, 5)

In [8]:
rfm_df[
    ["Recency", "Purchase_Frequency", "Monetary_Value"]
].describe()

,Recency,Purchase_Frequency,Monetary_Value
count,5878.000000,5878.000000,5878.000000
mean,201.331916,6.289384,2955.904095
std,209.338707,13.009406,14440.852688
min,1.000000,1.000000,2.950000
25%,26.000000,1.000000,342.280000
50%,96.000000,3.000000,867.740000
75%,380.000000,7.000000,2248.305000
max,739.000000,398.000000,580987.040000


In [9]:
rfm_df["R_Score"] = pd.qcut(
    rfm_df["Recency"],
    5,
    labels=[5,4,3,2,1]
)

rfm_df["F_Score"] = pd.qcut(
    rfm_df["Purchase_Frequency"].rank(method="first"),
    5,
    labels=[1,2,3,4,5]
)

rfm_df["M_Score"] = pd.qcut(
    rfm_df["Monetary_Value"].rank(method="first"),
    5,
    labels=[1,2,3,4,5]
)

rfm_df[
    [
        "Recency",
        "Purchase_Frequency",
        "Monetary_Value",
        "R_Score",
        "F_Score",
        "M_Score"
    ]
].head()

,Recency,Purchase_Frequency,Monetary_Value,R_Score,F_Score,M_Score
0,326,12,77556.46,2,5,5
1,2,8,4921.53,5,4,5
2,75,5,2019.40,3,4,4
3,19,4,4428.69,5,3,5
4,310,1,334.40,2,1,2


In [10]:
rfm_df["RFM_Score"] = (
    rfm_df["R_Score"].astype(str) +
    rfm_df["F_Score"].astype(str) +
    rfm_df["M_Score"].astype(str)
)

rfm_df[
    [
        "Customer ID",
        "R_Score",
        "F_Score",
        "M_Score",
        "RFM_Score"
    ]
].head()

,Customer ID,R_Score,F_Score,M_Score,RFM_Score
0,12346.0,2,5,5,255
1,12347.0,5,4,5,545
2,12348.0,3,4,4,344
3,12349.0,5,3,5,535
4,12350.0,2,1,2,212


In [11]:
rfm_df["Segment"] = "Others"

rfm_df.loc[
    (rfm_df["R_Score"] >= 4) &
    (rfm_df["F_Score"] >= 4) &
    (rfm_df["M_Score"] >= 4),
    "Segment"
] = "Champions"

rfm_df.loc[
    (rfm_df["R_Score"] >= 3) &
    (rfm_df["F_Score"] >= 3) &
    (rfm_df["M_Score"] >= 3) &
    (rfm_df["Segment"] != "Champions"),
    "Segment"
] = "Loyal Customers"

rfm_df.loc[
    (rfm_df["R_Score"] <= 2) &
    (rfm_df["F_Score"] >= 3),
    "Segment"
] = "At Risk"

rfm_df.loc[
    (rfm_df["R_Score"] <= 2) &
    (rfm_df["F_Score"] <= 2),
    "Segment"
] = "Lost Customers"

rfm_df["Segment"].value_counts()

Segment
At Risk            3254
Lost Customers     1449
Others              978
Loyal Customers     137
Champions            60
Name: count, dtype: int64

In [12]:
rfm_df.to_csv(
    "../../02_Cleaned_Data/customer_segments.csv",
    index=False
)

print("Customer Segmentation Saved Successfully")

Customer Segmentation Saved Successfully


In [13]:
import os

os.listdir("../../02_Cleaned_Data")

['customer_segments.csv', 'online_retail_cleaned.csv']

In [14]:
rfm_df.groupby("Segment").agg({
    "Recency":"mean",
    "Purchase_Frequency":"mean",
    "Monetary_Value":"mean"
}).round(2)

,Recency,Purchase_Frequency,Monetary_Value
Segment,,,
At Risk,90.09,10.04,4813.68
Champions,486.62,8.73,4115.12
Lost Customers,173.11,1.37,595.62
Loyal Customers,486.78,3.44,1349.75
Others,555.78,1.33,425.57


In [15]:
rfm_df[
    ["Recency","R_Score"]
].sample(20)

,Recency,R_Score
71,113,3
5293,17,5
5687,8,5
775,5,5
5664,374,2
705,78,3
5156,513,1
5450,6,5
1649,77,3
5819,12,5


# Customer Segmentation Summary

## Objective
Segment customers using RFM analysis.

## Key Findings
- Champions represent the most valuable customer group.
- Loyal customers contribute consistent business value.
- At-Risk customers require retention efforts.
- Lost Customers show low engagement and activity.

## Recommendations
- Reward Champions with loyalty benefits.
- Upsell Loyal Customers.
- Re-engage At-Risk customers through targeted campaigns.
- Limit acquisition spending on Lost Customers.